# Hypothesis Testing — COVID-19 Policies & Mobility

This notebook contains the hypothesis testing part of the project:

**COVID-19 Government Policies and Mobility Response**

We use the merged policy + mobility dataset:

**covid_merged_final.csv (country × date panel) containing**
- mobility measures (retail, grocery, parks, transit, workplaces, residential)
- policy indicators C1–C8 and E1
- stringency index
- country and date information

In the EDA notebook, I mainly explored patterns and visual relationships between
policies and mobility. In this notebook, the goal is to move from description to
statistical testing and check whether these relationships are supported by
significant evidence (α = 0.05, decisions based on p-values).

## Hypotheses Tested

**H1:** Policies and mobility are significantly associated at the global level.

**H2:** Mobility sectors move together rather than independently (spillover).

**H3:** The strength of the policy–mobility relationship differs across countries.

**H4:** The policy–mobility relationship changes across different phases of the pandemic.

In [82]:
import pandas as pd

# Load processed merged dataset
df = pd.read_csv("covid_merged_final.csv")

print("Shape:", df.shape)
print("\nFirst rows:")
display(df.head())

Shape: (3872, 21)

First rows:


,countrycode,country,date,retail,grocery,parks,transit,workplaces,residential,stringencyindex,...,c2_workplace_closing,c3_cancel_public_events,c4_restrictions_on_gatherings,c5_close_public_transport,c6_stay_at_home_requirements,c7_restrictions_on_internal_movement,c8_international_travel_controls,e1_income_support,confirmedcases,confirmeddeaths
0,BR,Brazil,2020-02-15,5.0,4.0,-5.0,8.0,6.0,0.0,11.11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
1,BR,Brazil,2020-02-16,2.0,3.0,-13.0,3.0,0.0,1.0,11.11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
2,BR,Brazil,2020-02-17,-2.0,0.0,-12.0,9.0,19.0,-1.0,11.11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
3,BR,Brazil,2020-02-18,-3.0,-1.0,-11.0,9.0,15.0,-1.0,11.11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
4,BR,Brazil,2020-02-19,-1.0,-2.0,-5.0,8.0,14.0,-1.0,11.11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN


In [83]:
# Column-wise averages (NaN values ignored)

col_means = df.mean(numeric_only=True, skipna=True)

print("Column-wise averages (NaNs excluded):")
display(col_means.round(3))


Column-wise averages (NaNs excluded):


retail                                    -26.769
grocery                                    -4.750
parks                                      17.537
transit                                   -30.632
workplaces                                -24.280
residential                                 9.006
stringencyindex                            61.587
c1_school_closing                           2.115
c2_workplace_closing                        1.765
c3_cancel_public_events                     1.750
c4_restrictions_on_gatherings               2.918
c5_close_public_transport                   0.588
c6_stay_at_home_requirements                1.172
c7_restrictions_on_internal_movement        1.349
c8_international_travel_controls            2.647
e1_income_support                           1.509
confirmedcases                          11534.248
confirmeddeaths                          7737.715
dtype: float64

## H1 — Policy Indicators and Mobility

In this step I test whether policy indicators and mobility variables are related in a
statistically meaningful way at the global level.

- **H0:** There is no relationship between policy indicators and mobility.
- **H1:** There is a statistically significant relationship between them.

I use Spearman correlation and interpret the result based on the p-value.

In [84]:
from scipy.stats import kendalltau
import pandas as pd

# --- select variables used in H1 ---

policy_vars = [
    "c2_workplace_closing",
    "c6_stay_at_home_requirements"
]

mobility_vars = [
    "workplaces",
    "retail",
    "transit",
    "residential"
]

# --- compute Kendall Tau for each pair ---

results_tau = []

for p in policy_vars:
    for m in mobility_vars:

        subset = df[[p, m]].dropna()

        tau, pval = kendalltau(subset[p], subset[m])

        results_tau.append({
            "policy": p,
            "mobility": m,
            "kendall_tau": round(tau, 3),
            "p_value": format(pval, ".3g")
        })

h1_tau_results = pd.DataFrame(results_tau)

display(h1_tau_results)


,policy,mobility,kendall_tau,p_value
0,c2_workplace_closing,workplaces,-0.380,4.39e-208
1,c2_workplace_closing,retail,-0.503,0
2,c2_workplace_closing,transit,-0.441,4.66e-280
3,c2_workplace_closing,residential,0.450,1.5e-283
4,c6_stay_at_home_requirements,workplaces,-0.364,4.25e-184
5,c6_stay_at_home_requirements,retail,-0.424,5.76e-250
6,c6_stay_at_home_requirements,transit,-0.414,6.25e-238
7,c6_stay_at_home_requirements,residential,0.369,1.07e-184


***Interpretation:***

The correlations are statistically significant for all tested pairs, so I reject H0 and conclude that policy indicators and mobility are statistically associated at the global level (association-based, not casual).

## H2 — Mobility Sectors

In this step I test whether different mobility sectors move together rather than independently.

- **H0:** There is no relationship between mobility sectors.
- **H1:** Mobility sectors are significantly related to each other.

I test this using Spearman correlation between sector pairs and interpret the results using the p-value.

In [85]:
from scipy.stats import spearmanr
import pandas as pd

# mobility variables to test
mobility_cols = [
    "workplaces",
    "retail",
    "transit",
    "grocery",
    "parks",
    "residential"
]

h2_results = []

for i in range(len(mobility_cols)):
    for j in range(i+1, len(mobility_cols)):

        m1 = mobility_cols[i]
        m2 = mobility_cols[j]

        subset = df[[m1, m2]].dropna()

        rho, pval = spearmanr(subset[m1], subset[m2])

        h2_results.append({
            "mobility_1": m1,
            "mobility_2": m2,
            "spearman_rho": round(rho, 3),
            "p_value": format(pval, ".3g")
        })

h2_results = pd.DataFrame(h2_results)
display(h2_results)


,mobility_1,mobility_2,spearman_rho,p_value
0,workplaces,retail,0.509,4.26e-254
1,workplaces,transit,0.786,0
2,workplaces,grocery,0.534,8.34e-285
3,workplaces,parks,0.144,1.81e-19
4,workplaces,residential,-0.792,0
5,retail,transit,0.802,0
6,retail,grocery,0.622,0
7,retail,parks,0.610,0
8,retail,residential,-0.761,0
9,transit,grocery,0.708,0


***Interpretation:***

Since the correlations between most mobility sectors are statistically significant, reject H0 and conclude that mobility sectors tend to move together rather than independently. 

## H3 — Cross-Country Differences

In this step I test whether the strength of the relationship between policies and mobility
differs across countries.

For H3, I focus on one key relationship used in the EDA:

C6 — stay-at-home requirements  
→ residential mobility

- **H0:** The policy–mobility relationship is the same across countries.
- **H1:** The strength of the policy–mobility relationship differs between countries.

I compute the correlation separately for each country and compare the results.

In [86]:
from scipy.stats import spearmanr
import pandas as pd

policy = "c6_stay_at_home_requirements"
mobility = "residential"

h3_results = []

for country, group in df.groupby("country"):

    subset = group[[policy, mobility]].dropna()

    if len(subset) > 5:  # avoid tiny samples
        rho, pval = spearmanr(subset[policy], subset[mobility])

        h3_results.append({
            "country": country,
            "spearman_rho": round(rho, 3),
            "p_value": format(pval, ".3g"),
            "n_obs": len(subset)
        })

h3_results = pd.DataFrame(h3_results).sort_values("spearman_rho", ascending=False)

display(h3_results)


,country,spearman_rho,p_value,n_obs
1,France,0.746,9.97e-64,352
2,Germany,0.727,3.63e-59,352
6,Spain,0.625,1.43e-39,352
9,United Kingdom,0.623,3.38e-39,352
10,United States,0.596,2.79e-35,352
0,Brazil,0.370,7.13e-13,352
7,Sweden,0.254,1.42e-06,352
3,Italy,0.217,4.19e-05,352
8,Turkey,0.181,0.000629,352
4,Japan,0.159,0.00276,352


***Interpretation:***

The results show that the policy–mobility relationship differs across countries, meaning similar policies led to stronger behavioral responses in some countries than others.

## H4 — Temporal Change in Relationship Strength (Three-Phase Comparison)

In this step, I test whether the relationship between policies and mobility changes across different stages of the pandemic instead of staying constant over time.

I use three phases:

- **Phase 1 — Early lockdown / first wave**
- **Phase 2 — Reopening and adjustment**
- **Phase 3 — Winter wave / later pandemic period**

For this test, I focus on three policy–mobility pairs:

- **C5 → Transit mobility** (public transport closures)
- **C7 → Retail mobility** (internal movement restrictions)
- **C1 → Parks mobility** (school closures and activity substitution)

For each phase, I compute the correlation for these pairs and compare how the strength of the relationship changes across phases.

In [87]:
import numpy as np
from scipy.stats import spearmanr
import pandas as pd

# make sure date is datetime
df["date"] = pd.to_datetime(df["date"])

phases = [
    ("Phase 1 (first lockdown)",      "2020-03-01", "2020-07-01"),
    ("Phase 2 (re-opening / adjustment)", "2020-07-01", "2020-11-01"),
    ("Phase 3 (winter wave)",         "2020-11-01", "2021-02-01"),
]

# policy–mobility pairs for H4 (no C2/C6)
pairs = [
    ("c5_close_public_transport",       "transit"),   # public transport closures
    ("c7_restrictions_on_internal_movement", "retail"),   # internal movement restrictions
    ("c1_school_closing",               "parks"),    # school closures & outdoor activity
]

h4_results = []

for phase_label, start_date, end_date in phases:
    phase_mask = (df["date"] >= start_date) & (df["date"] < end_date)
    df_phase = df.loc[phase_mask]

    for policy, mobility in pairs:
        subset = df_phase[[policy, mobility]].dropna()

        # skip if too few observations
        if len(subset) < 10:
            continue

        rho, pval = spearmanr(subset[policy], subset[mobility])

        h4_results.append({
            "phase": phase_label,
            "policy": policy,
            "mobility": mobility,
            "spearman_rho": round(rho, 3),
            "p_value": format(pval, ".3g"),
            "n_obs": len(subset)
        })

h4_results = pd.DataFrame(h4_results)
display(h4_results)


,phase,policy,mobility,spearman_rho,p_value,n_obs
0,Phase 1 (first lockdown),c5_close_public_transport,transit,-0.597,2.04e-130,1342
1,Phase 1 (first lockdown),c7_restrictions_on_internal_movement,retail,-0.601,1.21e-132,1342
2,Phase 1 (first lockdown),c1_school_closing,parks,-0.402,2.77e-53,1342
3,Phase 2 (re-opening / adjustment),c5_close_public_transport,transit,-0.367,1.85e-44,1353
4,Phase 2 (re-opening / adjustment),c7_restrictions_on_internal_movement,retail,-0.390,1.83e-50,1353
5,Phase 2 (re-opening / adjustment),c1_school_closing,parks,-0.084,0.00206,1353
6,Phase 3 (winter wave),c5_close_public_transport,transit,-0.121,0.000122,1012
7,Phase 3 (winter wave),c7_restrictions_on_internal_movement,retail,-0.093,0.00306,1012
8,Phase 3 (winter wave),c1_school_closing,parks,-0.108,0.000595,1012


***Interpretation:***

Since the correlation strengths decline from Phase 1 to Phase 3 (while remaining significant), I reject H0 and conclude that the policy–mobility relationship weakens across pandemic phases, consistent with adaptation over time.

**All statistical tests are correlation-based and rely on observational data, so results are interpreted as associations rather than causal effects.**